Task 15 — Enterprise LLM Gateway

The task uses FastAPI, Redis, Docker, Prometheus, and Grafana to create a gateway that handles rate limiting, model fallback, latency tracking, and request queuing.

In [2]:
# Import Libraries

!pip install redis
from fastapi import FastAPI
import time
import redis

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 560.6/560.6 kB 3.0 MB/s eta 0:00:00


In [3]:
# Create FastAPI App

app = FastAPI()

In [9]:
# Connect Redis

# Install and start Redis server
!apt-get update
!apt-get install redis-server -y
!service redis-server start

r = redis.Redis(
    host="localhost",
    port=6379,
    decode_responses=True
)

print("Redis connected")

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [105 kB]
Get:4 https://cli.github.com/packages stable/main amd64 Packages [356 B]
Get:5 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:9 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [3,164 kB]
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:11 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [4,190 kB]
Hit:12 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:13 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:14 https://pp

In [5]:
# Token Bucket Settings

MAX_TOKENS = 10
REFILL_RATE = 1

In [6]:
# Rate Limiting

def allow_request(user_id):

    key = f"tokens:{user_id}"

    tokens = r.get(key)

    if tokens is None:
        tokens = MAX_TOKENS
    else:
        tokens = int(tokens)

    if tokens > 0:
        tokens -= 1
        r.set(key, tokens)
        return True

    return False

In [10]:
# Test Rate Limiting

user = "user1"

print(allow_request(user))

True


In [8]:
# Primary Model

def primary_model(prompt):

    return f"Primary model response: {prompt}"

In [11]:
# Fallback Model

def fallback_model(prompt):

    return f"Fallback model response: {prompt}"

In [12]:
# Model Fallback

def generate_response(prompt):

    try:
        response = primary_model(prompt)
        return response

    except Exception:
        return fallback_model(prompt)

In [13]:
# Track Latency

start = time.time()

response = generate_response(
    "Explain artificial intelligence"
)

latency = time.time() - start

print("Response:", response)
print("Latency:", latency)

Response: Primary model response: Explain artificial intelligence
Latency: 0.00014209747314453125


In [14]:
# Gateway Endpoint

@app.get("/generate")
def generate(prompt: str):

    if not allow_request("user1"):
        return {"error": "Rate limit exceeded"}

    start = time.time()

    response = generate_response(prompt)

    latency = time.time() - start

    return {
        "response": response,
        "latency": latency
    }

In [15]:
# Results

print("Response:", response)
print("Latency:", latency)
print("Gateway running successfully")

Response: Primary model response: Explain artificial intelligence
Latency: 0.00014209747314453125
Gateway running successfully
